In [0]:
# DBTITLE 1,Get Parameters from Widgets
# Get parameters from widgets (passed by LoopConsolidationFiles)
FileId = dbutils.widgets.get("FileId")
CurrentContainer = dbutils.widgets.get("CurrentContainer")
CurrentFolderPath = dbutils.widgets.get("CurrentFolderPath")
ConsolidatedLayerDataModel = dbutils.widgets.get("ConsolidatedLayerDataModel")
ConsolidatedLayerDataModelFilePath = dbutils.widgets.get("ConsolidatedLayerDataModelFilePath")
ConsolidatedMappingFileName = dbutils.widgets.get("ConsolidatedMappingFileName")
ConsolidatedMappingFilePath = dbutils.widgets.get("ConsolidatedMappingFilePath")
ConsolidatedFolderPath = dbutils.widgets.get("ConsolidatedFolderPath")

# Construct paths - SERVERLESS COMPATIBLE (no /mnt/ mount points)
FullProcessed = f"{CurrentContainer}/{CurrentFolderPath}/"
FullConsolidatedFolderPath = ConsolidatedFolderPath
DataModelFile = f"{ConsolidatedLayerDataModelFilePath}/{ConsolidatedLayerDataModel}"
ConsolidationMapping = f"{ConsolidatedMappingFilePath}/{ConsolidatedMappingFileName}"

print("Parameters loaded")
print(f"  FileId: {FileId}")
print(f"  FullProcessed: {FullProcessed}")
print(f"  FullConsolidatedFolderPath: {FullConsolidatedFolderPath}")
print(f"  DataModelFile: {DataModelFile}")
print(f"  ConsolidationMapping: {ConsolidationMapping}")


In [0]:
# DBTITLE 1,Import Libraries
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import DataFrame, Row
from delta.tables import *
import json

print("Libraries imported")

In [0]:
# DBTITLE 1,Helper Functions - Type Mapping

def get_sql_type(data_type):
    """Convert DataModel data type to SQL type string"""
    type_mapping = {
        "StringType": "STRING",
        "IntegerType": "INT",
        "LongType": "BIGINT",
        "DoubleType": "DOUBLE",
        "FloatType": "FLOAT",
        "BooleanType": "BOOLEAN",
        "DateType": "DATE",
        "TimestampType": "TIMESTAMP",
        "DecimalType": "DECIMAL(38,10)"
    }
    return type_mapping.get(data_type, "STRING")

def get_data_type(data_type):
    """Convert DataModel data type to PySpark DataType"""
    type_mapping = {
        "StringType": StringType(),
        "IntegerType": IntegerType(),
        "LongType": LongType(),
        "DoubleType": DoubleType(),
        "FloatType": FloatType(),
        "BooleanType": BooleanType(),   
        "DateType": DateType(),
        "TimestampType": TimestampType(),
        "DecimalType": DecimalType(38, 10)
    }
    return type_mapping.get(data_type, StringType())

def get_struct(data_model_df):
    """Create StructType schema from DataModel DataFrame"""
    fields = []
    for row in data_model_df.orderBy("Ordinal").collect():
        field_name = row.FieldName
        data_type = get_data_type(row.DataType)
        fields.append(StructField(field_name, data_type, nullable=True))
    return StructType(fields)

print("Type mapping functions defined")

In [0]:
# DBTITLE 1,Helper Functions - Column Transformation

def get_select_expr(cols_df):
    """
    Build SQL select expressions with type casting
    Converts columns from source schema to destination schema with proper type casting
    Returns: List of select expression strings
    """
    sql_commands = []
    
    for row in cols_df.collect():
        field_name = row.FieldName
        data_type = row.DataType
        ordinal = row.Ordinal
        # Clean source column name - strip any extraneous quotes from mapping file
        source_column = row.SourceColumn.strip().strip("'").strip('"') if row.SourceColumn else ""
        destination_column = row.DestinationColumn
        source_column_format = row.SourceColumnFormat if row.SourceColumnFormat else ""
        # Clean column_query too - strip extraneous quotes
        column_query_raw = row.ColumnQuery if row.ColumnQuery else ""
        column_query = column_query_raw.strip().strip("'").strip('"') if column_query_raw else ""
        
        # Build the select expression based on the column type and format
        if column_query and column_query.strip():
            # If there's a custom query, use it
            expr = f"NULLIF(CAST({column_query} AS {get_sql_type(data_type)}),'') AS {destination_column}"
        elif data_type == "DateType" and source_column_format:
            # Date with format
            expr = f"to_date({source_column},'{source_column_format}') AS {destination_column}"
        elif data_type == "TimestampType" and source_column_format:
            # Timestamp with format
            expr = f"to_timestamp({source_column},'{source_column_format}') AS {destination_column}"
        else:
            # Standard type casting
            expr = f"CAST({source_column} AS {get_sql_type(data_type)}) AS {destination_column}"
        
        sql_commands.append(expr)
    
    return sql_commands

def custom_select(available_cols, required_cols):
    """
    Add null columns for missing fields from DataModel
    This ensures the output DataFrame matches the DataModel schema exactly
    """
    return [
        col(column) if column in available_cols else lit(None).alias(column)
        for column in required_cols
    ]

print("Column transformation functions defined")

In [0]:
# DBTITLE 1,Callable Function - process_consolidation
# This function can be called from LoopConsolidationFiles

def process_consolidation(FileId, CurrentContainer, CurrentFolderPath,
                         ConsolidatedLayerDataModel, ConsolidatedLayerDataModelFilePath,
                         ConsolidatedMappingFileName, ConsolidatedMappingFilePath,
                         ConsolidatedFolderPath):
    """
    Process a single file consolidation
    Called from LoopConsolidationFiles notebook
    
    Returns: JSON string with processing result
    """
    rJSON = {}
    double_quote = '"'
    
    try:
        # Get notebook context (Serverless compatible)
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            current_job_id = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
        except Exception:
            current_job_id = "Undefined"
        
        rJSON["CurrentJobId"] = current_job_id
        
        # Construct paths - SERVERLESS COMPATIBLE (no /mnt/ mount points)
        FullProcessed = f"{CurrentContainer}/{CurrentFolderPath}/"
        FullConsolidatedFolderPath = ConsolidatedFolderPath
        DataModelFile = f"{ConsolidatedLayerDataModelFilePath}/{ConsolidatedLayerDataModel}"
        ConsolidationMapping = f"{ConsolidatedMappingFilePath}/{ConsolidatedMappingFileName}"
        
        # Load DataModel JSON
        temp_data_model = spark.read.format("json").option("multiline", "true").load(DataModelFile)
        data_model = temp_data_model.select(explode(col("Fields"))).select(
            col("col.FieldName").alias("FieldName"),
            col("col.DataType").alias("DataType"),
            col("col.Ordinal").alias("Ordinal")
        )
        
        # Create destination schema
        dest_schema = get_struct(data_model)
        df_data_model = spark.createDataFrame([], dest_schema)
        
        # Load ConsolidationMapping file
        consolidated_mappings = spark.read.format("json").option("multiline", "true").load(ConsolidationMapping)
        temp_mappings = consolidated_mappings.select(explode(col("columnMapping"))).select(
            col("col.recordType").alias("recordType"),
            col("col.selectColumns").alias("selectColumns")
        )
        
        # Explode RecordType and selectColumns
        s_record_type = temp_mappings.select(explode(col("recordType"))).select(
            col("col.Field").alias("Field"),
            col("col.Value").alias("Value")
        )
        s_columns = temp_mappings.select(explode(col("selectColumns"))).select(
            col("col.SourceColumn").alias("SourceColumn"),
            col("col.DestinationColumn").alias("DestinationColumn"),
            col("col.SourceColumnFormat").alias("SourceColumnFormat"),
            col("col.ColumnQuery").alias("ColumnQuery")
        )
        
        # Merge mapping columns to datamodel
        seq_columns = data_model.join(
            s_columns,
            data_model["FieldName"] == s_columns["DestinationColumn"],
            "inner"
        ).select(
            data_model["FieldName"],
            data_model["DataType"],
            data_model["Ordinal"],
            s_columns["SourceColumn"],
            s_columns["DestinationColumn"],
            s_columns["SourceColumnFormat"],
            s_columns["ColumnQuery"]
        )
        
        # Build select expressions (list of sql queries)
        sel_cols = get_select_expr(seq_columns)
        
        # Create filter expression
        record_type_row = s_record_type.select("Field", "Value").collect()[-1]
        filter_field = "FileId" if not record_type_row.Field or record_type_row.Field == "" else record_type_row.Field
        filter_value = FileId if not record_type_row.Value or record_type_row.Value == "" else record_type_row.Value
        
        # Load and transform parquet file
        df_file_reformatted = spark.read.format("parquet").load(FullProcessed) \
            .selectExpr(*sel_cols) \
            .filter(col("FileID") == FileId) \
            .filter(col(filter_field) == filter_value)
        
        # Union with DataModel to enforce schema
        df_file = df_data_model.union(
            df_file_reformatted.select(
                *custom_select(
                    df_file_reformatted.columns,
                    df_data_model.columns
                )
            )
        )
        
        record_count = df_file.count()
        
        # Write to Delta if not empty
        if record_count > 0:
            df_file.write.format("delta").option("mergeSchema", "true").mode("append").save(FullConsolidatedFolderPath)
        
        # Build success response
        rJSON["ConsolidatedCount"] = str(record_count)
        rJSON["Status"] = "SUCCESS"
        rJSON["ErrorMessage"] = ""
        
    except Exception as e:
        # Handle errors
        error_msg = str(e).replace(double_quote, "").replace("\n", " ").replace("\r", " ").replace("\t", " ").strip()
        error_msg = ''.join(char for char in error_msg if ord(char) >= 32)
        
        rJSON["ConsolidatedCount"] = "0"
        rJSON["Status"] = "FAILED"
        rJSON["ErrorMessage"] = error_msg
    
    return json.dumps(rJSON)

print("process_consolidation function defined and ready to be called")